In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import gc
import ast
import copy
import time
import random
import numpy as np
import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as torchdata

from Model.MENDR.MENDREncoder import MENDREncoder, WaveletEncoderDecoder
from Model.MENDR.MENDRContextualizer import MENDRContextualizer
from Model.MENDR.MENDRTrainer import MENDRTrainer
from Model.MENDR.mAtt.optimizer import MixOptimizer

from dataset import WaveletFinetuningDataset
from torch_geometric.loader import DataLoader
from sklearn.metrics import balanced_accuracy_score
import optuna
from optuna.trial import TrialState

/home/hice1/mchen439/scratch/miniconda3/envs/Wavelet/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load Dataset
print("*" * 50)
train_dataset = WaveletFinetuningDataset(root="/home/hice1/mchen439/scratch/downstreamTaskData/train")
print("Dataset Loaded. Length of Train Dataset: ", len(train_dataset))
eval_dataset = WaveletFinetuningDataset(root="/home/hice1/mchen439/scratch/downstreamTaskData/eval")
print("Dataset Loaded. Length of Validation Dataset: ", len(eval_dataset))


**************************************************
Loading 2130 folders


100%|██████████| 2130/2130 [02:01<00:00, 17.50it/s]


Dataset Loaded. Length of Train Dataset:  44854
Loading 253 folders


100%|██████████| 253/253 [00:11<00:00, 21.85it/s]

Dataset Loaded. Length of Validation Dataset:  5101


In [4]:
### Seed ###
torch.cuda.empty_cache()
random.seed(42)
os.environ['PYTHONHASHSEED'] = str(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
## CUDNN ##
torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

In [5]:
BANDS = ['delta', 'theta', 'alpha', 'beta', 'gamma']
train_loader = DataLoader(train_dataset, batch_size=256, num_workers=2, shuffle=True)
eval_loader = DataLoader(eval_dataset, batch_size=256, num_workers=2, shuffle=False)
print(len(train_loader), len(eval_loader))

176 20


In [6]:
def define_model(trial):
    # We optimize the number of layers, hidden units, dropout ratio in each layer.
    n_layers = trial.suggest_int("n_layers", 1, 3)
    layers = []
    in_features = 570

    for i in range(n_layers):
        out_features = trial.suggest_int("n_units_l{}".format(i), 95, 380)
        layers.append(nn.Linear(in_features, out_features))
        layers.append(nn.ReLU())
        layers.append(nn.BatchNorm1d(out_features))
        p = trial.suggest_float("dropout_l{}".format(i), 0.2, 0.5)
        layers.append(nn.Dropout(p))
        in_features = out_features

    layers.append(nn.Linear(in_features, 2))
    layers.append(nn.LogSoftmax(dim=1))

    return nn.Sequential(*layers)

In [44]:
### Model ###
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = MENDREncoder(device=device)
contextualizer = MENDRContextualizer(device=device)

for param in encoder.parameters():
    param.requires_grad = False

for param in contextualizer.parameters():
    param.requires_grad = False

encoder.load_state_dict(torch.load("checkpoint/batch_size256_4epochs/encoder_weights.pth"))
contextualizer.load_state_dict(torch.load("checkpoint/batch_size256_4epochs/contextualizer_weights.pth"))

#optim_params = []
#optim_params += list(encoder.parameters())
#optim_params += list(contextualizer.parameters())
#optim_params += list(decoder.parameters())

#optimizer = torch.optim.AdamW(optim_params, lr=0.001)
#scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=50, threshold=.1, mode='min')
#optimizer = MixOptimizer(optimizer)


<All keys matched successfully>

In [45]:
# Quick hack
contextualizer.wavelet_r2e_lin = contextualizer.wavelet_r2e_lin.to(device)
contextualizer.wavelet_combined_r2e_lin = contextualizer.combined_r2e_lin.to(device)

In [46]:
def _std_norm(x):
	mean = torch.mean(x, dim=(0, 1), keepdim=True)
	std = torch.std(x, dim=(0, 1), keepdim=True)
	x = (x - mean) / std
	return x

In [47]:
def train_one_epoch(encoder, contextualizer, decoder, optimizer, scheduler, task_loss_fn, train_loader):
	loss_sum = 0
	# Freeze foundation weights
	encoder.eval()
	contextualizer.train()
	# Only train decoder
	decoder.train()

	pbar = tqdm.trange(len(train_loader), desc="Training")
	data_iterator = iter(train_loader)
	balanced_accuracies = []
	for iteration in pbar:
		data = next(data_iterator)

		for key, value in data.items():
			if isinstance(value, torch.Tensor):
				# Perform batch normalization across channels 
				data[key] = _std_norm(value.float().to(device))

		relevant_bands = [data['delta'].float().to(device),
		 				  data['theta'].float().to(device),
						  data['alpha'].float().to(device),
						  data['beta'].float().to(device),
						  data['gamma'].float().to(device)]

		inputs = dict(zip(BANDS, relevant_bands))

		with torch.no_grad():
			encoder_outputs = encoder(data['graph'], inputs)
			combined_r2e_output, combined_manifold_output, wavelet_r2e_output, wavelet_manifold_output = contextualizer(encoder_outputs)

		embed_cat = [combined_r2e_output]
		for band, embedding in wavelet_r2e_output.items():
			embed_cat.append(embedding)
		embed_cat = torch.cat(embed_cat, dim=1)

		prediction = decoder(embed_cat)
		ground_truth = data['graph'].y.to(device)

		task_loss = task_loss_fn(prediction, ground_truth)

		#print(torch.argmax(prediction, axis=1).shape)

		y_pred = torch.argmax(prediction, axis=1).cpu().numpy()
		y_true = ground_truth.cpu().numpy()

		correct = (torch.argmax(prediction, axis=1) == ground_truth).sum().item()
		accuracy = correct / prediction.size(0)
		balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
		balanced_accuracies.append(balanced_accuracy)
		loss = task_loss

		scheduler.step(iteration)

		optimizer.zero_grad()
		loss.backward()
		optimizer.step()
		loss_sum += loss.item()

		pbar.set_postfix(task_loss=task_loss.item(), accuracy=f'{accuracy*100:.3f}', balanced_accuracy=f'{balanced_accuracy*100:.3f}', lr=scheduler.get_last_lr()[0])
	balanced_accuracies = np.array(balanced_accuracies)	
	print(f'Mean Acc: {balanced_accuracies.mean()} Acc Std: {balanced_accuracies.std()}')
	return loss_sum

In [48]:
def validate_one_epoch(encoder, contextualizer, decoder, task_loss_fn, val_loader):
	loss_sum = 0
	encoder.eval()
	contextualizer.eval()
	decoder.eval()

	pbar = tqdm.trange(len(val_loader), desc="Validating")
	data_iterator = iter(val_loader)
	balanced_accuracies = []
	with torch.no_grad():
		for iteration in pbar:
			data = next(data_iterator)
			
			for key, value in data.items():
				if isinstance(value, torch.Tensor):
					# Perform batch normalization across channels 
					data[key] = _std_norm(value.float().to(device))

			relevant_bands = [data['delta'].float().to(device),
							data['theta'].float().to(device),
							data['alpha'].float().to(device),
							data['beta'].float().to(device),
							data['gamma'].float().to(device)]

			inputs = dict(zip(BANDS, relevant_bands))

			encoder_outputs = encoder(data['graph'], inputs)
			combined_r2e_output, combined_manifold_output, wavelet_r2e_output, wavelet_manifold_output = contextualizer(encoder_outputs)

			embed_cat = [combined_r2e_output]
			for band, embedding in wavelet_r2e_output.items():
				embed_cat.append(embedding)
			embed_cat = torch.cat(embed_cat, dim=1)

			prediction = decoder(embed_cat)
			ground_truth = data['graph'].y.to(device)

			task_loss = task_loss_fn(prediction, ground_truth)

			loss = task_loss
			loss_sum += loss.item()

			correct = (torch.argmax(prediction, axis=1) == ground_truth).sum().item()
			accuracy = correct / prediction.size(0)

			y_pred = torch.argmax(prediction, axis=1).cpu().numpy()
			y_true = ground_truth.cpu().numpy()
			balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
			balanced_accuracies.append(balanced_accuracy)

			pbar.set_postfix(
				task_loss=task_loss.item(),
				accuracy=f'{accuracy*100:.3f}',
				balanced_accuracy=f'{balanced_accuracy*100:.3f}')

		balanced_accuracies = np.array(balanced_accuracies)
		print(f'Mean Acc: {balanced_accuracies.mean()} Acc Std: {balanced_accuracies.std()}')
		return loss_sum, balanced_accuracies.mean(), balanced_accuracies.std()

In [49]:
def objective(trial):
	decoder = define_model(trial).to(device)
	optimizer_name = trial.suggest_categorical("optimizer", ["Adam"])
	lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
	optimizer = getattr(optim, optimizer_name)(decoder.parameters(), lr=lr)
	scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=100, T_mult=2)
	for epoch_idx in range(1):
		training_loss = train_one_epoch(encoder, contextualizer, decoder, optimizer, scheduler, nn.NLLLoss(), train_loader)
		eval_loss, eval_acc, eval_std = validate_one_epoch(encoder, contextualizer, decoder, nn.NLLLoss(), eval_loader)
	return eval_acc, eval_std
	#	print("Epoch: ", epoch_idx, "Total Training Loss: ", training_loss, "Total Validation Loss:", eval_loss)

In [50]:
study = optuna.create_study(directions=["maximize", "minimize"])
study.optimize(objective, n_trials=20, n_jobs=4)

[I 2025-02-15 23:44:44,232] A new study created in memory with name: no-name-bc39516f-641d-4d10-a5a7-86147bdd2ffc
Training:   0%|          | 0/176 [00:00<?, ?it/s]


Training:   1%|          | 1/176 [00:14<41:54, 14.37s/it, accuracy=48.438, balanced_accuracy=48.512, lr=0.0283, task_loss=0.827]





Training:   1%|          | 2/176 [00:25<36:28, 12.58s/it, accuracy=58.594, balanced_accuracy=57.898, lr=0.0283, task_loss=0.909]





Training:   2%|▏         | 3/176 [00:37<34:57, 12.12s/it, accuracy=54.297, balanced_accuracy=53.224, lr=0.0283, task_loss=1.18] 





Training:   2%|▏         | 4/176 [00:48<34:00, 11.86s/it, accuracy=54.688, balanced_accuracy=53.311, lr=0.0282, task_loss=0.997]





Training:   3%|▎         | 5/176 [01:00<33:13, 11.66s/it, accuracy=60.156, balanced_accuracy=58.735, lr=0.0282, task_loss=0.811]





Training:   3%|▎         | 6/176 [01:11<33:11, 11.71s/it, accuracy=58.203, balanced_accuracy=57.083, lr=0.0281, task_loss=0.817]





Training:   4%|▍         | 7/1

Mean Acc: 0.5655053461179455 Acc Std: 0.03685444634275188








Training: 100%|██████████| 176/176 [33:46<00:00, 11.52s/it, accuracy=50.000, balanced_accuracy=50.137, lr=0.000141, task_loss=0.756]


Mean Acc: 0.5733737538518701 Acc Std: 0.04026458630627433




Training:  99%|█████████▉| 174/176 [33:48<00:22, 11.48s/it, accuracy=67.578, balanced_accuracy=64.290, lr=0.02, task_loss=0.603]  

...





Validating: 100%|██████████| 20/20 [03:39<00:00, 10.97s/it, accuracy=56.118, balanced_accuracy=63.247, task_loss=0.694]
[I 2025-02-16 02:13:38,104] Trial 15 finished with values: [0.6602869423050627, 0.1131900772387913] and parameters: {'n_layers': 2, 'n_units_l0': 100, 'dropout_l0': 0.32350998717659596, 'n_units_l1': 289, 'dropout_l1': 0.3817043653939655, 'optimizer': 'Adam', 'lr': 0.003535768014705066}.


Mean Acc: 0.6608073947730635 Acc Std: 0.11170275859688818


Training:   0%|          | 0/176 [00:00<?, ?it/s][I 2025-02-16 02:13:38,674] Trial 14 finished with values: [0.6608073947730635, 0.11170275859688818] and parameters: {'n_layers': 3, 'n_units_l0': 197, 'dropout_l0': 0.44995428580412855, 'n_units_l1': 165, 'dropout_l1': 0.3517784887567109, 'n_units_l2': 276, 'dropout_l2': 0.4030095151499531, 'optimizer': 'Adam', 'lr': 0.00121969118211695}.




Training:   1%|          | 1/176 [00:13<39:26, 13.52s/it, accuracy=46.875, balanced_accuracy=46.627, lr=2.09e-5, task_loss=0.839]





Training:   1%|          | 2/176 [00:25<35:44, 12.33s/it, accuracy=56.250, balanced_accuracy=55.189, lr=2.09e-5, task_loss=0.782]





Training:   2%|▏         | 3/176 [00:36<34:07, 11.83s/it, accuracy=55.859, balanced_accuracy=55.969, lr=2.09e-5, task_loss=0.736]





Training:   2%|▏         | 4/176 [00:47<33:28, 11.68s/it, accuracy=55.078, balanced_accuracy=54.956, lr=2.09e-5, task_loss=0.793]





Training:   3%|▎         | 5/176 [00:59<33:02, 11.59s/it, accurac

KeyboardInterrupt: 

Training:  48%|████▊     | 84/176 [16:02<17:34, 11.46s/it, accuracy=49.609, balanced_accuracy=49.511, lr=1.46e-6, task_loss=0.797]



In [ ]:
pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

print("Study statistics: ")
print("  Number of finished trials: ", len(study.trials))
print("  Number of pruned trials: ", len(pruned_trials))
print("  Number of complete trials: ", len(complete_trials))

print("Best trial:")
trials = study.best_trials

for trial in trials:
    print("  Values: ", trial.values)

    print("  Params: ")
    for key, value in trial.params.items():
        print("    {}: {}".format(key, value))

Study statistics: 
  Number of finished trials:  20
  Number of pruned trials:  0
  Number of complete trials:  20
Best trial:
  Values:  [0.501001074132004, 0.009629024263738717]
  Params: 
    n_layers: 1
    n_units_l0: 214
    dropout_l0: 0.23201696464371624
    optimizer: Adam
    lr: 0.005004000305036596
  Values:  [0.5103697613307782, 0.033453182291278145]
  Params: 
    n_layers: 1
    n_units_l0: 256
    dropout_l0: 0.2321412149238731
    optimizer: Adam
    lr: 0.004382845060405519
  Values:  [0.5022939889549425, 0.010078262237321632]
  Params: 
    n_layers: 3
    n_units_l0: 272
    dropout_l0: 0.3185324410797712
    n_units_l1: 220
    dropout_l1: 0.22507496353250092
    n_units_l2: 216
    dropout_l2: 0.3144843304786751
    optimizer: Adam
    lr: 0.0007882571038524167
  Values:  [0.5011300688193875, 0.009943548426417505]
  Params: 
    n_layers: 2
    n_units_l0: 130
    dropout_l0: 0.4016931484390791
    n_units_l1: 273
    dropout_l1: 0.20743682469986224
    optimizer:

In [37]:
class optimalModel(nn.Module):
    def __init__(self):
        super(optimalModel, self).__init__()
        self.layer0 = nn.Sequential(nn.Linear(570, 57), nn.ReLU(), nn.BatchNorm1d(57), nn.Dropout(p=0.1))
        #self.layer1 = nn.Sequential(nn.Linear(219, 272), nn.ReLU(), nn.BatchNorm1d(272), nn.Dropout(p=0.44111219955831804))
        #self.layer2 = nn.Sequential(nn.Linear(272, 296), nn.ReLU(), nn.BatchNorm1d(296), nn.Dropout(p=0.30320965604443806))
        self.out = nn.Sequential(nn.Linear(57, 2), nn.LogSoftmax(dim=1))

    def forward(self, x):
        x = self.layer0(x)
        #x = self.layer1(x)
        #x = self.layer2(x)
        return self.out(x)

decoder = optimalModel().to(device)
optimizer = optim.Adam(decoder.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=80, T_mult=2)
print("Total Params:", sum(p.numel() for p in decoder.parameters() if p.requires_grad))

Total Params: 32777


In [43]:
for epoch_idx in range(20):
    training_loss = train_one_epoch(encoder, contextualizer, decoder, optimizer, scheduler, nn.NLLLoss(), train_loader)
    eval_loss, eval_acc, eval_std = validate_one_epoch(encoder, contextualizer, decoder, nn.NLLLoss(), eval_loader)
    print("*" * 50)
    print(f"Epoch: {epoch_idx}")
    print("Training Loss:", training_loss)
    print(f"Eval Loss: {eval_loss} Eval Mean Acc: {eval_acc} Eval Std: {eval_std}")

Training: 100%|██████████| 176/176 [09:49<00:00,  3.35s/it, accuracy=64.815, balanced_accuracy=63.736, lr=0.000355, task_loss=0.643]


Mean Acc: 0.572076318658689 Acc Std: 0.0354069187844129


Validating: 100%|██████████| 20/20 [00:58<00:00,  2.90s/it, accuracy=37.131, balanced_accuracy=54.012, task_loss=0.999]


Mean Acc: 0.5187192387147042 Acc Std: 0.03118302400941642
**************************************************
Epoch: 0
Training Loss: 116.87079590559006
Eval Loss: 14.140398800373077 Eval Mean Acc: 0.5187192387147042 Eval Std: 0.03118302400941642


Training: 100%|██████████| 176/176 [09:53<00:00,  3.37s/it, accuracy=62.963, balanced_accuracy=60.227, lr=0.000355, task_loss=0.677]


...

Mean Acc: 0.5832971562999677 Acc Std: 0.037866190291351916


Validating: 100%|██████████| 20/20 [00:57<00:00,  2.90s/it, accuracy=37.131, balanced_accuracy=54.012, task_loss=1.01] 


Mean Acc: 0.5283587148530494 Acc Std: 0.03849527624632049
**************************************************
Epoch: 5
Training Loss: 115.80887496471405
Eval Loss: 13.996208012104034 Eval Mean Acc: 0.5283587148530494 Eval Std: 0.03849527624632049


Training:   2%|▏         | 3/176 [00:11<11:27,  3.98s/it, accuracy=62.109, balanced_accuracy=60.441, lr=0.000998, task_loss=0.66]


KeyboardInterrupt: 